# WSJCAM0 Autoregressive Speech Prediction

This notebook assumes the server environment is already prepared.

Before running this notebook:

1. install dependencies from the terminal by following `INSTALL_SERVER.md`
2. switch the notebook kernel to `Python (finalproject26-py39)`
3. run the cells below


In [ ]:
from pathlib import Path
import os
import warnings

warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'src').exists() else NOTEBOOK_DIR.parent
os.chdir(ROOT)
print(f'Project root: {ROOT}')
print(f'Dataset root exists: {(ROOT / "wsj0").exists()}')


In [ ]:
HF_ENDPOINT = 'https://hf-mirror.com'
os.environ['HF_ENDPOINT'] = HF_ENDPOINT
os.environ['HUGGINGFACE_HUB_ENDPOINT'] = HF_ENDPOINT
print('HF_ENDPOINT =', os.environ['HF_ENDPOINT'])


In [ ]:
import sys

import pandas as pd
import torch
import torchaudio

print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Torchaudio:', torchaudio.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from src.data.manifest import build_wsj0_manifest

train_manifest = build_wsj0_manifest('wsj0', 'si_tr_s')
val_manifest = build_wsj0_manifest('wsj0', 'si_dt_05')
test_manifest = build_wsj0_manifest('wsj0', 'si_et_05')
pd.DataFrame([
    {'split': 'si_tr_s', 'num_files': len(train_manifest)},
    {'split': 'si_dt_05', 'num_files': len(val_manifest)},
    {'split': 'si_et_05', 'num_files': len(test_manifest)},
])


In [ ]:
from src.experiment import compare_codecs, run_codec_experiment

SMOKE_RUN = False
DATASET_ROOT = 'wsj0'
ARTIFACTS_DIR = 'artifacts'

COMMON_OVERRIDES = {
    'dataset': {
        'root': DATASET_ROOT,
    },
    'project': {
        'artifacts_dir': ARTIFACTS_DIR,
    },
    'codecs': {
        'hub_endpoint': HF_ENDPOINT,
    },
}

if SMOKE_RUN:
    COMMON_OVERRIDES = {
        **COMMON_OVERRIDES,
        'training': {
            'batch_size': 2,
            'num_workers': 0,
            'max_epochs': 1,
            'context_frames': 40,
            'log_every_epochs': 1,
        },
        'evaluation': {
            'max_eval_files': 8,
            'save_audio_examples': 2,
        },
    }

COMMON_OVERRIDES


In [ ]:
facodec_config, facodec_train, facodec_eval = run_codec_experiment('facodec', overrides=COMMON_OVERRIDES)
print(facodec_train)
print(facodec_eval)


In [ ]:
encodec_config, encodec_train, encodec_eval = run_codec_experiment('encodec', overrides=COMMON_OVERRIDES)
print(encodec_train)
print(encodec_eval)


In [ ]:
summary_table = compare_codecs({
    'facodec': facodec_eval.summary_path,
    'encodec': encodec_eval.summary_path,
})
summary_table.to_csv(Path(ARTIFACTS_DIR) / 'codec_comparison.csv', index=False)
summary_table
